# Student Performance — End-to-End Machine Learning

Download the Student Performance CSV from the LMS **Study Material** tab. Run the single code cell below in Google Colab and upload the dataset when prompted.

In [ ]:
# Student Performance ML workflow — one Google Colab cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('Please upload the Student Performance CSV file.')

df = pd.read_csv(csv_files[0])
df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='deep')

def find_column(candidates):
    normalized = {col.lower().replace('_', ' ').replace('-', ' ').strip(): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for name, original in normalized.items():
        if any(candidate in name for candidate in candidates):
            return original
    return None

target = find_column(['exam score', 'exam_score'])
if target is None:
    raise KeyError(f'Could not find exam_score. Available columns: {df.columns.tolist()}')
df[target] = pd.to_numeric(df[target], errors='coerce')
df = df.dropna(subset=[target]).drop_duplicates().copy()

# Correct likely numeric performance fields if the CSV stored them as text.
numeric_keywords = ['hour', 'attendance', 'frequency', 'rating', 'score', 'age', 'percentage', 'attempt', 'class']
for column in df.columns:
    if column == target or any(word in column.lower() for word in numeric_keywords):
        converted = pd.to_numeric(df[column].astype(str).str.replace(r'[^0-9.-]', '', regex=True), errors='coerce')
        if converted.notna().sum() >= max(1, df[column].notna().sum() * 0.5):
            df[column] = converted

print('=' * 95)
print(f'STUDENT PERFORMANCE — END-TO-END MACHINE LEARNING: {csv_files[0]}')
print('=' * 95)

# 1. EDA
print('\n1. EXPLORATORY DATA ANALYSIS')
print(f'Dataset shape after removing missing targets/duplicates: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
print(f'Numeric variables ({len(numeric_columns)}): {numeric_columns}')
print(f'Categorical variables ({len(categorical_columns)}): {categorical_columns}')
print('\nMissing values:')
display(df.isna().sum().to_frame(name='Missing Values').sort_values('Missing Values', ascending=False))
print('\nDescriptive statistics:')
display(df.describe(include='all').T)

plt.figure(figsize=(8, 5))
sns.histplot(df[target], bins=25, kde=True, color='#2a6fbb')
plt.title('Distribution of Exam Score')
plt.xlabel('Exam Score')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()

# Relationships between requested study/lifestyle variables and exam_score.
requested_features = [
    find_column(['study hours per day']), find_column(['attendance percentage']), find_column(['sleep hours']),
    find_column(['social media hours']), find_column(['netflix hours']), find_column(['exercise frequency']),
    find_column(['mental health rating'])
]
requested_features = [column for column in requested_features if column]
numeric_relationships = [column for column in requested_features if column in numeric_columns]
if numeric_relationships:
    rows = int(np.ceil(len(numeric_relationships) / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(12, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for axis, column in zip(axes, numeric_relationships):
        sns.regplot(data=df, x=column, y=target, scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'}, ax=axis)
        axis.set_title(f'{column} vs {target}')
    for axis in axes[len(numeric_relationships):]:
        axis.set_visible(False)
    plt.tight_layout()
    plt.show()

correlation = df[numeric_columns].corr()
target_correlations = correlation[target].drop(target).sort_values(key=abs, ascending=False)
print('\nNumerical correlations with exam_score:')
display(target_correlations.to_frame(name='Correlation with Exam Score').round(3))
if len(numeric_columns) >= 2:
    plt.figure(figsize=(max(10, len(numeric_columns) * 0.7), max(7, len(numeric_columns) * 0.6)))
    sns.heatmap(correlation, cmap='coolwarm', center=0, annot=False, square=True)
    plt.title('Correlation Heatmap of Numerical Variables')
    plt.tight_layout()
    plt.show()

# Potential outliers: IQR count for numeric predictors.
outlier_counts = {}
for column in [c for c in numeric_columns if c != target]:
    q1, q3 = df[column].quantile(0.25), df[column].quantile(0.75)
    iqr = q3 - q1
    outlier_counts[column] = int(((df[column] < q1 - 1.5 * iqr) | (df[column] > q3 + 1.5 * iqr)).sum()) if iqr > 0 else 0
display(pd.Series(outlier_counts, name='Potential IQR Outliers').sort_values(ascending=False).to_frame())

# 2. PREPARE FEATURES — identifiers are excluded; all other usable features are retained.
print('\n2. DATA PREPARATION')
id_columns = [column for column in df.columns if column.lower() in ['id', 'student_id', 'unnamed:_0'] or column.lower().endswith('_id')]
feature_columns = [column for column in df.columns if column not in [target] + id_columns]
X = df[feature_columns].copy()
y_regression = df[target].copy()
y_classification = (df[target] >= 50).astype(int)
print(f'Regression target: {target}')
print('Classification target: Pass (1) if exam_score ≥ 50; Fail (0) otherwise')
print(f'Features used ({len(feature_columns)}): {feature_columns}')
print(f'Class balance — Pass: {y_classification.mean()*100:.2f}%, Fail: {(1-y_classification.mean())*100:.2f}%')

# A single split is used for both tasks. Stratification preserves Pass/Fail proportions.
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_regression, y_classification, test_size=0.20, random_state=42, stratify=y_classification
)
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [column for column in X_train.columns if column not in numeric_features]
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('numeric', numeric_pipeline, numeric_features), ('categorical', categorical_pipeline, categorical_features)])
print(f'Train/test split: {len(X_train):,} / {len(X_test):,}')
print('Preprocessing is fit on training data only inside each model pipeline, preventing data leakage.')

# 3. LINEAR REGRESSION
print('\n3. REGRESSION: LINEAR REGRESSION FOR EXAM SCORE')
regression_model = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
regression_model.fit(X_train, y_reg_train)
reg_train_pred = regression_model.predict(X_train)
reg_test_pred = regression_model.predict(X_test)
def regression_metrics(actual, predicted):
    return [mean_absolute_error(actual, predicted), mean_squared_error(actual, predicted), np.sqrt(mean_squared_error(actual, predicted)), r2_score(actual, predicted)]
regression_results = pd.DataFrame([regression_metrics(y_reg_train, reg_train_pred), regression_metrics(y_reg_test, reg_test_pred)], index=['Training', 'Testing'], columns=['MAE', 'MSE', 'RMSE', 'R²']).round(3)
display(regression_results)
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_reg_test, y=reg_test_pred, alpha=0.7)
limits = [min(y_reg_test.min(), reg_test_pred.min()), max(y_reg_test.max(), reg_test_pred.max())]
plt.plot(limits, limits, 'r--', label='Perfect prediction')
plt.title('Linear Regression: Actual vs Predicted Exam Score (Test Set)')
plt.xlabel('Actual Exam Score')
plt.ylabel('Predicted Exam Score')
plt.legend()
plt.tight_layout()
plt.show()

# Most influential Linear Regression features by absolute standardized coefficient.
feature_names = regression_model.named_steps['preprocessor'].get_feature_names_out()
coefficients = pd.Series(regression_model.named_steps['model'].coef_, index=feature_names).sort_values(key=abs, ascending=False)
print('Top Linear Regression coefficients (absolute size):')
display(coefficients.head(10).to_frame(name='Coefficient').round(3))

# 4. CLASSIFICATION
print('\n4. CLASSIFICATION: PASS/FAIL (PASS = EXAM SCORE ≥ 50)')
classification_model = Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])
classification_model.fit(X_train, y_cls_train)
cls_train_pred = classification_model.predict(X_train)
cls_test_pred = classification_model.predict(X_test)
def classification_metrics(actual, predicted):
    return [accuracy_score(actual, predicted), precision_score(actual, predicted, zero_division=0), recall_score(actual, predicted, zero_division=0), f1_score(actual, predicted, zero_division=0)]
classification_results = pd.DataFrame([classification_metrics(y_cls_train, cls_train_pred), classification_metrics(y_cls_test, cls_test_pred)], index=['Training', 'Testing'], columns=['Accuracy', 'Precision', 'Recall', 'F1-score']).round(3)
display(classification_results)
matrix = confusion_matrix(y_cls_test, cls_test_pred)
ConfusionMatrixDisplay(matrix, display_labels=['Fail', 'Pass']).plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()
print(f'Test-set confusion matrix: TN={matrix[0,0]}, FP={matrix[0,1]}, FN={matrix[1,0]}, TP={matrix[1,1]}')

# 5. MODEL COMPARISON, FIT DIAGNOSIS, AND INSIGHTS
print('\n5. MODEL COMPARISON AND FIT DIAGNOSIS')
reg_gap = regression_results.loc['Training', 'R²'] - regression_results.loc['Testing', 'R²']
cls_gap = classification_results.loc['Training', 'Accuracy'] - classification_results.loc['Testing', 'Accuracy']
reg_fit = 'possible overfitting' if reg_gap > 0.10 else ('possible underfitting' if regression_results.loc['Testing', 'R²'] < 0.20 else 'reasonable generalization')
cls_fit = 'possible overfitting' if cls_gap > 0.05 else ('possible underfitting' if classification_results.loc['Testing', 'Accuracy'] < 0.60 else 'reasonable generalization')
print(f'Regression R² gap (train − test): {reg_gap:.3f} → {reg_fit}.')
print(f'Classification accuracy gap (train − test): {cls_gap:.3f} → {cls_fit}.')

top_correlation_feature = target_correlations.index[0] if not target_correlations.empty else 'No comparable numeric feature'
top_correlation_value = target_correlations.iloc[0] if not target_correlations.empty else np.nan
top_coefficient = coefficients.index[0] if not coefficients.empty else 'No coefficient available'
observations = [
    f'1. The numeric feature most strongly correlated with exam_score is {top_correlation_feature} (r = {top_correlation_value:.3f}).',
    f'2. Linear Regression achieved a test R² of {regression_results.loc["Testing", "R²"]:.3f} with a test RMSE of {regression_results.loc["Testing", "RMSE"]:.3f}.',
    f'3. The strongest Linear Regression coefficient by magnitude is {top_coefficient}, after scaling and encoding.',
    f'4. The Pass/Fail model achieved test accuracy of {classification_results.loc["Testing", "Accuracy"]:.3f} and F1-score of {classification_results.loc["Testing", "F1-score"]:.3f}.',
    f'5. Regression shows {reg_fit}; classification shows {cls_fit}, based on training-versus-testing performance gaps.'
]
print('\nFIVE MEANINGFUL INSIGHTS')
for observation in observations:
    print(observation)